# Gold Price Analysis Notebook

This notebook analyzes Vietnamese gold price data and prepares daily insights for downstream dashboard/chatbot use.

Main goals:

1. Load and audit gold price data
2. Clean timestamp and numeric fields
3. Analyze price coverage by product/type
4. Plot price trends and technical indicators
5. Detect high-volatility days
6. Compare domestic gold products with XAUUSD
7. Optionally join with enriched news data

## 0. Environment Setup

Run this section first in Google Colab.

In [ ]:
!pip install pandas numpy matplotlib seaborn python-dateutil

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Environment is ready.")

## 1. Load Data

Upload `gold_price.csv` to Colab before running this cell.

Expected columns:

- `ts`
- `type_code`
- `brand`
- `gold_type`
- `buy_price`
- `sell_price`
- `mid_price`
- `spread`
- `ema20`, `ema50`, `rsi14`, `macd`, `macd_signal`, `macd_hist`
- `source_site`
- `created_at`

In [ ]:
DATA_PATH = "/content/gold_price.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"File not found: {DATA_PATH}. Please upload gold_price.csv to Colab Files."
    )

price_df = pd.read_csv(DATA_PATH)

print("Rows:", len(price_df))
print("Columns:", len(price_df.columns))
price_df.head()

## 2. Initial Audit

Check schema, missing values, date coverage, and product coverage.

In [ ]:
def missing_like_count(series):
    missing_na = series.isna()
    missing_text = series.astype(str).str.strip().isin(["", "nan", "None", "[]"])
    return int((missing_na | missing_text).sum())

schema_df = pd.DataFrame({
    "column": price_df.columns,
    "dtype": [price_df[col].dtype for col in price_df.columns],
    "missing_count": [missing_like_count(price_df[col]) for col in price_df.columns],
    "missing_rate": [round(missing_like_count(price_df[col]) / len(price_df), 4) for col in price_df.columns],
})

schema_df

In [ ]:
price_df["ts"] = pd.to_datetime(price_df["ts"], errors="coerce")
price_df["created_at"] = pd.to_datetime(price_df["created_at"], errors="coerce")

price_audit = {
    "total_rows": len(price_df),
    "min_ts": price_df["ts"].min(),
    "max_ts": price_df["ts"].max(),
    "unique_timestamps": price_df["ts"].nunique(),
    "unique_type_codes": price_df["type_code"].nunique(),
    "missing_ts": int(price_df["ts"].isna().sum()),
    "missing_mid_price": int(price_df["mid_price"].isna().sum()),
    "duplicate_rows_by_ts_type": int(price_df.duplicated(subset=["ts", "type_code"], keep=False).sum()),
}

pd.DataFrame([price_audit]).T.rename(columns={0: "value"})

In [ ]:
coverage_by_type = (
    price_df
    .groupby(["type_code", "brand", "gold_type"], dropna=False)
    .agg(
        rows=("ts", "count"),
        min_ts=("ts", "min"),
        max_ts=("ts", "max"),
        avg_buy=("buy_price", "mean"),
        avg_sell=("sell_price", "mean"),
        avg_mid=("mid_price", "mean"),
        avg_spread=("spread", "mean"),
    )
    .reset_index()
    .sort_values(["brand", "type_code"])
)

for col in ["avg_buy", "avg_sell", "avg_mid", "avg_spread"]:
    coverage_by_type[col] = coverage_by_type[col].round(2)

coverage_by_type

## 3. Clean and Feature Engineering

Create date-level fields and daily returns.

In [ ]:
price_clean = price_df.copy()

numeric_cols = [
    "buy_price", "sell_price", "mid_price", "spread",
    "ema20", "ema50", "rsi14", "macd", "macd_signal", "macd_hist",
]

for col in numeric_cols:
    if col in price_clean.columns:
        price_clean[col] = pd.to_numeric(price_clean[col], errors="coerce")

price_clean = price_clean.dropna(subset=["ts", "type_code", "mid_price"]).copy()
price_clean = price_clean.sort_values(["type_code", "ts"])

price_clean["date"] = price_clean["ts"].dt.date
price_clean["daily_return"] = price_clean.groupby("type_code")["mid_price"].pct_change()
price_clean["daily_return_pct"] = price_clean["daily_return"] * 100
price_clean["price_change"] = price_clean.groupby("type_code")["mid_price"].diff()
price_clean["spread_pct"] = np.where(
    price_clean["mid_price"] != 0,
    price_clean["spread"] / price_clean["mid_price"] * 100,
    np.nan,
)

price_clean.head()

## 4. Price Trend by Product

Plot raw mid-price trends. Because XAUUSD has a different scale from domestic VND prices, we also create a normalized index chart later.

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=price_clean, x="ts", y="mid_price", hue="type_code", marker="o")
plt.title("Gold Mid Price Trend by Type Code")
plt.xlabel("Date")
plt.ylabel("Mid Price")
plt.xticks(rotation=45)
plt.legend(title="Type Code")
plt.tight_layout()
plt.show()

## 5. Normalized Price Index

Set the first available price of each product to `100`. This makes different price scales comparable.

In [ ]:
price_clean["base_mid_price"] = price_clean.groupby("type_code")["mid_price"].transform("first")
price_clean["price_index_100"] = price_clean["mid_price"] / price_clean["base_mid_price"] * 100

plt.figure(figsize=(14, 6))
sns.lineplot(data=price_clean, x="ts", y="price_index_100", hue="type_code", marker="o")
plt.title("Normalized Gold Price Index by Type Code — Base = 100")
plt.xlabel("Date")
plt.ylabel("Price Index")
plt.xticks(rotation=45)
plt.legend(title="Type Code")
plt.tight_layout()
plt.show()

## 6. Domestic Gold vs XAUUSD

Separate local VND products and international XAUUSD to avoid scale distortion.

In [ ]:
domestic_df = price_clean[price_clean["type_code"] != "XAUUSD"].copy()
xauusd_df = price_clean[price_clean["type_code"] == "XAUUSD"].copy()

print("Domestic rows:", len(domestic_df))
print("XAUUSD rows:", len(xauusd_df))

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=domestic_df, x="ts", y="mid_price", hue="type_code", marker="o")
plt.title("Domestic Gold Mid Price Trend")
plt.xlabel("Date")
plt.ylabel("Mid Price (VND-based source units)")
plt.xticks(rotation=45)
plt.legend(title="Type Code")
plt.tight_layout()
plt.show()

In [ ]:
if len(xauusd_df) > 0:
    plt.figure(figsize=(14, 5))
    sns.lineplot(data=xauusd_df, x="ts", y="mid_price", marker="o")
    plt.title("XAUUSD Mid Price Trend")
    plt.xlabel("Date")
    plt.ylabel("XAUUSD")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No XAUUSD data found.")

## 7. Technical Indicators

Plot available indicators: EMA20, EMA50, RSI14, MACD, MACD signal, and MACD histogram.

Choose one `type_code` first.

In [ ]:
selected_type_code = "SJL1L10"  # Change this value if needed

selected_df = price_clean[price_clean["type_code"] == selected_type_code].copy()

print("Selected type_code:", selected_type_code)
print("Rows:", len(selected_df))
selected_df.head()

In [ ]:
if len(selected_df) > 0:
    plt.figure(figsize=(14, 6))
    plt.plot(selected_df["ts"], selected_df["mid_price"], marker="o", label="Mid Price")

    if "ema20" in selected_df.columns:
        plt.plot(selected_df["ts"], selected_df["ema20"], label="EMA20")

    if "ema50" in selected_df.columns:
        plt.plot(selected_df["ts"], selected_df["ema50"], label="EMA50")

    plt.title(f"Price and EMA Indicators — {selected_type_code}")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data for selected type_code.")

In [ ]:
if len(selected_df) > 0 and "rsi14" in selected_df.columns:
    plt.figure(figsize=(14, 4))
    plt.plot(selected_df["ts"], selected_df["rsi14"], marker="o", label="RSI14")
    plt.axhline(70, linestyle="--", label="Overbought 70")
    plt.axhline(30, linestyle="--", label="Oversold 30")
    plt.title(f"RSI14 — {selected_type_code}")
    plt.xlabel("Date")
    plt.ylabel("RSI14")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No RSI14 data available.")

In [ ]:
required_macd_cols = {"macd", "macd_signal", "macd_hist"}

if len(selected_df) > 0 and required_macd_cols.issubset(selected_df.columns):
    plt.figure(figsize=(14, 5))
    plt.plot(selected_df["ts"], selected_df["macd"], marker="o", label="MACD")
    plt.plot(selected_df["ts"], selected_df["macd_signal"], marker="o", label="MACD Signal")
    plt.bar(selected_df["ts"], selected_df["macd_hist"], label="MACD Hist")
    plt.title(f"MACD Indicators — {selected_type_code}")
    plt.xlabel("Date")
    plt.ylabel("MACD")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("MACD columns are not available.")

## 8. Daily Return and Volatility Analysis

Detect days with large price moves. This will later help connect price movement with news events.

In [ ]:
return_summary = (
    price_clean
    .groupby("type_code")
    .agg(
        avg_return_pct=("daily_return_pct", "mean"),
        min_return_pct=("daily_return_pct", "min"),
        max_return_pct=("daily_return_pct", "max"),
        volatility_pct=("daily_return_pct", "std"),
        avg_spread_pct=("spread_pct", "mean"),
    )
    .reset_index()
)

for col in ["avg_return_pct", "min_return_pct", "max_return_pct", "volatility_pct", "avg_spread_pct"]:
    return_summary[col] = return_summary[col].round(4)

return_summary.sort_values("volatility_pct", ascending=False)

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=price_clean, x="ts", y="daily_return_pct", hue="type_code", marker="o")
plt.axhline(0, linestyle="--")
plt.title("Daily Return Percentage by Type Code")
plt.xlabel("Date")
plt.ylabel("Daily Return (%)")
plt.xticks(rotation=45)
plt.legend(title="Type Code")
plt.tight_layout()
plt.show()

In [ ]:
# Detect unusual moves using absolute daily return threshold
MOVE_THRESHOLD_PCT = 1.0

large_moves = price_clean[
    price_clean["daily_return_pct"].abs() >= MOVE_THRESHOLD_PCT
][
    [
        "ts", "date", "type_code", "brand", "gold_type",
        "mid_price", "price_change", "daily_return_pct",
        "spread", "spread_pct", "rsi14", "macd_hist",
    ]
].sort_values(["date", "daily_return_pct"], ascending=[False, False])

print("Large move rows:", len(large_moves))
large_moves.head(50)

## 9. Spread Analysis

The spread is the difference between sell and buy price. High spread can indicate market stress, illiquidity, or fast repricing.

In [ ]:
spread_summary = (
    price_clean
    .groupby(["type_code", "brand", "gold_type"], dropna=False)
    .agg(
        avg_spread=("spread", "mean"),
        max_spread=("spread", "max"),
        avg_spread_pct=("spread_pct", "mean"),
        max_spread_pct=("spread_pct", "max"),
    )
    .reset_index()
)

for col in ["avg_spread", "max_spread", "avg_spread_pct", "max_spread_pct"]:
    spread_summary[col] = spread_summary[col].round(4)

spread_summary.sort_values("avg_spread_pct", ascending=False)

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=price_clean, x="ts", y="spread_pct", hue="type_code", marker="o")
plt.title("Spread Percentage by Type Code")
plt.xlabel("Date")
plt.ylabel("Spread (%)")
plt.xticks(rotation=45)
plt.legend(title="Type Code")
plt.tight_layout()
plt.show()

## 10. Correlation Between Products

Use normalized daily returns to compare product co-movement.

In [ ]:
return_pivot = price_clean.pivot_table(
    index="date",
    columns="type_code",
    values="daily_return_pct",
    aggfunc="mean",
)

corr_matrix = return_pivot.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f")
plt.title("Correlation of Daily Returns by Type Code")
plt.tight_layout()
plt.show()

corr_matrix

## 11. Optional: Join with Enriched News Data

Upload `gold_news_enriched.csv` if available.

This section aggregates news sentiment and impact by date, then joins it with price data.

In [ ]:
NEWS_PATH = "/content/gold_news_enriched.csv"

if os.path.exists(NEWS_PATH):
    news_df = pd.read_csv(NEWS_PATH)
    news_df["published_at"] = pd.to_datetime(news_df["published_at"], errors="coerce")
    news_df["date"] = news_df["published_at"].dt.date

    for col in ["sentiment_score", "impact_score", "relevance_score", "quality_score"]:
        if col in news_df.columns:
            news_df[col] = pd.to_numeric(news_df[col], errors="coerce")

    print("News rows:", len(news_df))
    display(news_df.head())
else:
    news_df = None
    print("No enriched news file found. Upload gold_news_enriched.csv to run this section.")

In [ ]:
if news_df is not None:
    news_filtered = news_df.copy()

    if "is_relevant" in news_filtered.columns:
        # Handles both boolean and string boolean values
        news_filtered["is_relevant_bool"] = news_filtered["is_relevant"].astype(str).str.lower().isin(["true", "1", "yes"])
        news_filtered = news_filtered[news_filtered["is_relevant_bool"]]

    daily_news = (
        news_filtered
        .groupby("date")
        .agg(
            news_count=("id", "count") if "id" in news_filtered.columns else ("title", "count"),
            avg_sentiment=("sentiment_score", "mean"),
            avg_impact=("impact_score", "mean"),
            max_impact=("impact_score", "max"),
            avg_relevance=("relevance_score", "mean"),
        )
        .reset_index()
    )

    for col in ["avg_sentiment", "avg_impact", "max_impact", "avg_relevance"]:
        daily_news[col] = daily_news[col].round(4)

    display(daily_news.head())
else:
    print("Skipping daily news aggregation.")

In [ ]:
if news_df is not None:
    daily_price = (
        price_clean
        .groupby(["date", "type_code"])
        .agg(
            mid_price=("mid_price", "mean"),
            daily_return_pct=("daily_return_pct", "mean"),
            spread_pct=("spread_pct", "mean"),
        )
        .reset_index()
    )

    price_news = daily_price.merge(daily_news, on="date", how="left")

    display(price_news.head())
else:
    price_news = None
    print("Skipping price-news join.")

In [ ]:
if news_df is not None and price_news is not None:
    selected_type_code_for_join = selected_type_code
    plot_df = price_news[price_news["type_code"] == selected_type_code_for_join].copy()

    plt.figure(figsize=(14, 6))
    plt.plot(plot_df["date"], plot_df["mid_price"], marker="o", label="Mid Price")

    high_news_days = plot_df[plot_df["max_impact"].fillna(0) >= 0.70]
    plt.scatter(
        high_news_days["date"],
        high_news_days["mid_price"],
        s=120,
        label="High-impact news day",
    )

    plt.title(f"Price with High-impact News Markers — {selected_type_code_for_join}")
    plt.xlabel("Date")
    plt.ylabel("Mid Price")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipping price-news plot.")

## 12. Generate Insight Tables

Create tables that can be used for reporting or chatbot seed examples.

In [ ]:
latest_date = price_clean["date"].max()
latest_snapshot = price_clean[price_clean["date"] == latest_date].copy()

latest_snapshot = latest_snapshot[
    [
        "date", "type_code", "brand", "gold_type",
        "buy_price", "sell_price", "mid_price", "spread", "spread_pct",
        "daily_return_pct", "rsi14", "macd_hist",
    ]
].sort_values("type_code")

latest_snapshot

In [ ]:
# Top moving product-days by absolute daily return
top_moves = price_clean.dropna(subset=["daily_return_pct"]).copy()
top_moves["abs_return_pct"] = top_moves["daily_return_pct"].abs()

top_moves = top_moves[
    [
        "date", "type_code", "brand", "gold_type", "mid_price",
        "price_change", "daily_return_pct", "abs_return_pct", "spread_pct",
        "rsi14", "macd_hist",
    ]
].sort_values("abs_return_pct", ascending=False)

top_moves.head(30)

In [ ]:
analysis_summary = {
    "total_rows": len(price_clean),
    "date_min": str(price_clean["date"].min()),
    "date_max": str(price_clean["date"].max()),
    "type_codes": sorted(price_clean["type_code"].dropna().unique().tolist()),
    "largest_abs_move_pct": round(float(top_moves["abs_return_pct"].max()), 4) if len(top_moves) else None,
    "avg_domestic_spread_pct": round(float(domestic_df["spread_pct"].mean()), 4) if len(domestic_df) else None,
    "avg_xauusd_spread_pct": round(float(xauusd_df["spread_pct"].mean()), 4) if len(xauusd_df) else None,
}

analysis_summary

## 13. Export Analysis Outputs

Export cleaned price data and key summary tables.

In [ ]:
OUTPUT_DIR = "/content/gold_price_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

price_clean.to_csv(f"{OUTPUT_DIR}/gold_price_cleaned.csv", index=False, encoding="utf-8-sig")
coverage_by_type.to_csv(f"{OUTPUT_DIR}/coverage_by_type.csv", index=False, encoding="utf-8-sig")
return_summary.to_csv(f"{OUTPUT_DIR}/return_summary.csv", index=False, encoding="utf-8-sig")
spread_summary.to_csv(f"{OUTPUT_DIR}/spread_summary.csv", index=False, encoding="utf-8-sig")
latest_snapshot.to_csv(f"{OUTPUT_DIR}/latest_snapshot.csv", index=False, encoding="utf-8-sig")
top_moves.head(100).to_csv(f"{OUTPUT_DIR}/top_moves.csv", index=False, encoding="utf-8-sig")

if news_df is not None and price_news is not None:
    daily_news.to_csv(f"{OUTPUT_DIR}/daily_news_summary.csv", index=False, encoding="utf-8-sig")
    price_news.to_csv(f"{OUTPUT_DIR}/price_news_joined.csv", index=False, encoding="utf-8-sig")

print("Exported files to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))